# Análise Visual: TransNetV2 + YOLO + Labels do SoccerNet

Este notebook:
1. Instala as dependências necessárias
2. Baixa um jogo (vídeo curto) e o arquivo `Labels-v2.json` do SoccerNet (dados oficiais de eventos do jogo)
3. Corta o vídeo em um trecho pequeno para teste rápido
4. Roda TransNetV2 (detecção de cenas) e YOLO (detecção de objetos: player, referee, ball, goalkeeper)
5. Gera um **vídeo de saída pequeno e anotado** (caixas do YOLO + número da cena + contagem de objetos)
6. Gera 3 gráficos:
   - Gráfico 1: jogadores detectados x tempo (linha)
   - Gráfico 2: timeline dos cortes de cena do TransNetV2
   - Gráfico 3: os dois combinados, cruzando ainda com os eventos reais do jogo (gols, cartões etc.) do `Labels-v2.json`

> ⚠️ Este notebook foi montado para você rodar no Kaggle/Colab com GPU. Ele **não foi executado aqui**.

## 1. Instalação de dependências

In [1]:
# Bibliotecas necessárias para todo o pipeline
!pip install -q SoccerNet transnetv2-pytorch ultralytics opencv-python-headless av pandas numpy matplotlib seaborn


## 2. Imports e fix de compatibilidade do PyTorch

In [2]:
import json
import os
from functools import partial

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from transnetv2_pytorch import TransNetV2

# Fix para PyTorch 2.6+: weights_only mudou o default para True, quebrando modelos ultralytics
_original_torch_load = torch.load
torch.load = partial(_original_torch_load, weights_only=False)

from ultralytics import YOLO

sns.set_theme(style="whitegrid")


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 3. Download do vídeo e do dataset de labels do SoccerNet

O SoccerNet, além dos vídeos, disponibiliza por jogo um arquivo **`Labels-v2.json`** com a
anotação oficial dos eventos da partida (gol, cartão amarelo/vermelho, substituição, escanteio, etc.),
cada um com o campo `gameTime` (ex: `"1 - 02:35"` = 1º tempo, 02:35) e `position` (tempo em milissegundos
desde o início do tempo de jogo).

Isso é ótimo para **validação**: depois de gerar os gráficos de detecção, dá pra sobrepor os eventos
reais e ver se os picos de atividade (jogadores, cortes de cena) batem com os momentos de gol/cartão.

Para baixar, você precisa ter aceitado o NDA do SoccerNet em https://www.soccer-net.org/ (a senha
oficial de download é fornecida depois disso — abaixo está a senha pública usada nos exemplos do
próprio dataset, mas confirme a sua conforme o cadastro).

In [3]:
from SoccerNet.Downloader import SoccerNetDownloader

myDownloader = SoccerNetDownloader(LocalDirectory="videos")
myDownloader.password = "s0cc3rn3t"  # senha do NDA do SoccerNet

# Troque pelo jogo que quiser analisar (formato: "liga/temporada/data - hora time1 x time2")
GAME = "england_epl/2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley"

# 1) Vídeo do jogo (224p aqui só para testes rápidos e leves; use 720p se quiser mais qualidade)
# myDownloader.downloadGames(
#     files=["1_224p.mkv"],
#     split=["train", "valid", "test"],
#    #  games=[GAME],
# )

# 2) Labels-v2.json: eventos oficiais anotados do jogo (gol, cartão, etc.)
myDownloader.downloadGames(
    files=["Labels-v2.json"],
    split=["train", "valid", "test"],
    # games=[GAME],
)


## 4. Cortar o vídeo em um trecho curto (ex: 3 minutos) para teste

Isso evita rodar o pipeline inteiro em uma partida completa só para validar a ideia.

In [4]:
VIDEO_ENTRADA = "videos/england_epl/2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/1_224p.mkv"
VIDEO_CORTADO = "trecho_teste.mp4"
DURACAO_TESTE_SEG = 60 * 30

def cortar_video(caminho_entrada, caminho_saida, duracao_seg):
    cap = cv2.VideoCapture(caminho_entrada)
    if not cap.isOpened():
        raise FileNotFoundError(f"Não foi possível abrir o vídeo: {caminho_entrada}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames_desejado = int(duracao_seg * fps)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(caminho_saida, fourcc, fps, (width, height))

    frame_idx = 0
    while frame_idx < total_frames_desejado:
        ret, frame = cap.read()
        if not ret:
            break
        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"✅ Trecho de {frame_idx} frames ({frame_idx / fps:.1f}s) salvo em: {caminho_saida}")
    return fps

FPS_VIDEO = cortar_video(VIDEO_ENTRADA, VIDEO_CORTADO, DURACAO_TESTE_SEG)


FileNotFoundError: Não foi possível abrir o vídeo: videos/england_epl/2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/1_224p.mkv

## 5. Mapeamento de cenas com TransNetV2

In [ ]:
def mapear_cenas_transnet(video_path: str, device: str):
    """Lê o vídeo em baixíssima resolução e mapeia cenas com TransNetV2.
    Retorna: mapa_frames_cenas (lista por frame) e lista de cenas com start/end."""
    print("🎬 Mapeando transições de cenas com TransNetV2...")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Não foi possível abrir o vídeo: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

    tiny_frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (48, 27), interpolation=cv2.INTER_AREA)
        tiny_frames.append(frame_resized)
    cap.release()

    print(f"🧠 Rodando TransNetV2 ({len(tiny_frames)} frames)...")
    frames_tensor = torch.from_numpy(np.array(tiny_frames, dtype=np.uint8))
    model = TransNetV2()
    if device == "cuda":
        model = model.cuda()
        frames_tensor = frames_tensor.to(device)

    single_frame_pred, _ = model.predict_frames(frames_tensor, quiet=True)
    single_frame_np = single_frame_pred.cpu().detach().numpy()
    scenes = model.predictions_to_scenes_with_data(single_frame_np, fps=fps, threshold=0.5)

    mapa_frames_cenas = [1] * total_frames
    cenas_info = []
    for scene in scenes:
        shot_id = int(scene["shot_id"])
        start = int(scene["start_frame"])
        end = int(scene["end_frame"])
        for f in range(start, min(end + 1, total_frames)):
            mapa_frames_cenas[f] = shot_id
        cenas_info.append({
            "cena": shot_id,
            "start_frame": start,
            "end_frame": min(end, total_frames - 1),
            "start_seg": start / fps,
            "end_seg": min(end, total_frames - 1) / fps,
        })

    print(f"📊 {len(cenas_info)} cenas identificadas!")
    return mapa_frames_cenas, cenas_info, fps

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
mapa_frames_cenas, cenas_info, FPS_VIDEO = mapear_cenas_transnet(VIDEO_CORTADO, DEVICE)
df_cenas = pd.DataFrame(cenas_info)
df_cenas.head()


## 6. Detecção com YOLO + geração do vídeo de saída anotado

Aqui rodamos o YOLO frame a frame, desenhamos as caixas detectadas + um overlay de texto
(cena atual, contagem de jogadores/bola/árbitro/goleiro) e escrevemos tudo em um **vídeo de
saída pequeno** (resolução reduzida) para facilitar a conferência visual.

Também coletamos, por frame, o `tempo_seg` e as contagens — isso vira a base dos gráficos.

In [ ]:
CAMINHO_MODELO_YOLO = "/kaggle/input/datasets/giovannimorassi/best-pt/best.pt"  # ajuste para o caminho do seu modelo treinado
LARGURA_SAIDA = 640  # reduz a resolução do vídeo de saída para deixar o arquivo pequeno

CORES_CLASSE = {
    "player": (0, 200, 0),
    "referee": (0, 165, 255),
    "ball": (0, 0, 255),
    "goalkeeper": (255, 0, 0),
}

def carregar_modelo_yolo(model_path: str, conf: float = 0.25, iou: float = 0.45) -> YOLO:
    if os.path.isdir(model_path):
        arquivos_pt = [f for f in os.listdir(model_path) if f.endswith(".pt")]
        if arquivos_pt:
            model_path = os.path.join(model_path, arquivos_pt[0])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = YOLO(model_path)
    model.overrides["conf"] = conf
    model.overrides["iou"] = iou
    model.overrides["max_det"] = 1000
    model.to(device)
    print(f"🚀 YOLO carregado em: {device.upper()}")
    return model


def processar_video_com_overlay(video_path, mapa_frames_cenas, model, saida_path, largura_saida=640):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    largura_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    escala = largura_saida / largura_orig
    altura_saida = int(altura_orig * escala)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(saida_path, fourcc, fps, (largura_saida, altura_saida))

    registros = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        cena_atual = mapa_frames_cenas[frame_idx] if frame_idx < len(mapa_frames_cenas) else -1
        result = model.predict(frame, verbose=False, quantize=16)[0]

        contagens = {"player": 0, "referee": 0, "ball": 0, "goalkeeper": 0}
        ball_conf_max = 0.0

        frame_redim = cv2.resize(frame, (largura_saida, altura_saida))

        if result.boxes is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            cls_ids = result.boxes.cls.cpu().numpy().astype(int)

            for box, conf, cls_id in zip(boxes, confs, cls_ids):
                cls_name = model.names.get(cls_id, str(cls_id))
                contagens[cls_name] = contagens.get(cls_name, 0) + 1
                if cls_name == "ball":
                    ball_conf_max = max(ball_conf_max, float(conf))

                x1, y1, x2, y2 = (box * escala).astype(int)
                cor = CORES_CLASSE.get(cls_name, (200, 200, 200))
                cv2.rectangle(frame_redim, (x1, y1), (x2, y2), cor, 1)
                cv2.putText(frame_redim, f"{cls_name} {conf:.2f}", (x1, max(y1 - 4, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.35, cor, 1)

        # Overlay com informações gerais do frame
        texto_overlay = (
            f"Cena: {cena_atual} | Jogadores: {contagens['player']} | "
            f"Bola: {contagens['ball']} (conf {ball_conf_max:.2f}) | "
            f"Arbitro: {contagens['referee']} | Goleiro: {contagens['goalkeeper']}"
        )
        cv2.rectangle(frame_redim, (0, 0), (largura_saida, 20), (0, 0, 0), -1)
        cv2.putText(frame_redim, texto_overlay, (5, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

        writer.write(frame_redim)

        registros.append({
            "frame": frame_idx,
            "tempo_seg": frame_idx / fps,
            "cena": cena_atual,
            "player_count": contagens["player"],
            "ball_count": contagens["ball"],
            "ball_conf_max": ball_conf_max,
            "referee_count": contagens["referee"],
            "goalkeeper_count": contagens["goalkeeper"],
        })

        frame_idx += 1
        if frame_idx % 200 == 0:
            print(f"  -> processados {frame_idx} frames...")

    cap.release()
    writer.release()
    print(f"✅ Vídeo anotado salvo em: {saida_path}")
    return pd.DataFrame(registros)


modelo_yolo = carregar_modelo_yolo(CAMINHO_MODELO_YOLO)
VIDEO_SAIDA = "video_anotado_pequeno.mp4"
df_frames = processar_video_com_overlay(VIDEO_CORTADO, mapa_frames_cenas, modelo_yolo, VIDEO_SAIDA, LARGURA_SAIDA)
df_frames.head()


## 7. Carregar os eventos reais do jogo (`Labels-v2.json`)

Convertendo o `gameTime` (`"1 - 02:35"`) para segundos, e filtrando apenas os eventos que caem
dentro da janela de tempo do nosso trecho de teste (`DURACAO_TESTE_SEG`).

In [ ]:
CAMINHO_LABELS = "videos/england_epl/2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/Labels-v2.json"

def carregar_eventos_soccernet(caminho_labels, duracao_max_seg, tempo_do_jogo=1):
    """Lê o Labels-v2.json e retorna um DataFrame com os eventos do 'tempo_do_jogo' (1 ou 2)
    que caem dentro dos primeiros 'duracao_max_seg' segundos."""
    with open(caminho_labels, "r", encoding="utf-8") as f:
        dados = json.load(f)

    eventos = []
    for anotacao in dados.get("annotations", []):
        game_time = anotacao.get("gameTime", "")  # ex: "1 - 02:35"
        try:
            tempo_str, relogio = game_time.split(" - ")
            tempo_jogo = int(tempo_str)
            minutos, segundos = map(int, relogio.split(":"))
            tempo_seg = minutos * 60 + segundos
        except (ValueError, AttributeError):
            continue

        if tempo_jogo != tempo_do_jogo or tempo_seg > duracao_max_seg:
            continue

        eventos.append({
            "tempo_seg": tempo_seg,
            "label": anotacao.get("label"),
            "time": anotacao.get("team"),
            "visibilidade": anotacao.get("visibility"),
        })

    return pd.DataFrame(eventos).sort_values("tempo_seg").reset_index(drop=True)

df_eventos = carregar_eventos_soccernet(CAMINHO_LABELS, DURACAO_TESTE_SEG, tempo_do_jogo=1)
df_eventos


## 8. Gráfico 1 — Jogadores detectados x tempo do vídeo

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_frames["tempo_seg"], df_frames["player_count"], color="royalblue", linewidth=1.2)

for _, evento in df_eventos.iterrows():
    ax.axvline(evento["tempo_seg"], color="red", linestyle="--", alpha=0.6)
    ax.text(evento["tempo_seg"], ax.get_ylim()[1] * 0.95, evento["label"],
            rotation=90, fontsize=8, color="red", va="top")

ax.set_title("Jogadores detectados ao longo do tempo (linhas vermelhas = eventos reais do SoccerNet)")
ax.set_xlabel("Tempo (segundos)")
ax.set_ylabel("Nº de jogadores detectados")
plt.tight_layout()
plt.show()


## 9. Gráfico 2 — Timeline dos cortes de cena (TransNetV2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 2.5))

cores_ciclo = sns.color_palette("tab20", n_colors=max(len(df_cenas), 1))
barras = [(row["start_seg"], row["end_seg"] - row["start_seg"]) for _, row in df_cenas.iterrows()]
cores_barras = [cores_ciclo[i % len(cores_ciclo)] for i in range(len(df_cenas))]

ax.broken_barh(barras, (0, 1), facecolors=cores_barras, edgecolors="black", linewidth=0.3)

for _, evento in df_eventos.iterrows():
    ax.axvline(evento["tempo_seg"], color="red", linestyle="--", alpha=0.8)

ax.set_yticks([])
ax.set_xlabel("Tempo (segundos)")
ax.set_title("Timeline dos cortes de cena detectados pelo TransNetV2 (linhas vermelhas = eventos reais)")
ax.set_xlim(0, DURACAO_TESTE_SEG)
plt.tight_layout()
plt.show()


## 10. Gráfico 3 — Visão combinada: jogadores + cortes de cena + eventos reais

Aqui os dois gráficos anteriores são combinados em uma única figura com eixo X compartilhado,
facilitando ver se picos de jogadores coincidem com trocas de cena e com eventos reais do jogo.

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(14, 7), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}
)

# --- Painel de cima: jogadores detectados ---
ax1.plot(df_frames["tempo_seg"], df_frames["player_count"], color="royalblue", linewidth=1.2, label="Jogadores detectados")
ax1.set_ylabel("Nº de jogadores")
ax1.set_title("Análise combinada: jogadores, cortes de cena e eventos reais do jogo")

for _, evento in df_eventos.iterrows():
    ax1.axvline(evento["tempo_seg"], color="red", linestyle="--", alpha=0.7)
    ax1.text(evento["tempo_seg"], ax1.get_ylim()[1] * 0.95, evento["label"],
              rotation=90, fontsize=8, color="red", va="top")

# --- Painel de baixo: timeline de cenas do TransNetV2 ---
cores_ciclo = sns.color_palette("tab20", n_colors=max(len(df_cenas), 1))
barras = [(row["start_seg"], row["end_seg"] - row["start_seg"]) for _, row in df_cenas.iterrows()]
cores_barras = [cores_ciclo[i % len(cores_ciclo)] for i in range(len(df_cenas))]
ax2.broken_barh(barras, (0, 1), facecolors=cores_barras, edgecolors="black", linewidth=0.3)

for _, evento in df_eventos.iterrows():
    ax2.axvline(evento["tempo_seg"], color="red", linestyle="--", alpha=0.7)

ax2.set_yticks([])
ax2.set_xlabel("Tempo (segundos)")
ax2.set_ylabel("Cenas\n(TransNetV2)")

plt.tight_layout()
plt.show()


## Próximos passos

- Ajuste `GAME`, `CAMINHO_MODELO_YOLO` e `DURACAO_TESTE_SEG` conforme seu ambiente.
- O vídeo de saída (`video_anotado_pequeno.mp4`) já sai em resolução reduzida para ficar leve —
  ajuste `LARGURA_SAIDA` se quiser ainda menor.
- Com o `df_frames` já montado, dá para plugar diretamente as ideias de filtro discutidas antes
  (z-score de picos, classificação de cena aberta/fechada, janelas deslizantes, etc.).
- Os eventos do `Labels-v2.json` servem de "gabarito" para você validar se os filtros propostos
  realmente capturam os momentos certos antes de decidir o que vai para a LLM.